# Chronos-2 LoRA: M6 quantile inference, rounds 1 to 12

Runs the LoRA fine-tuned Chronos-2 from `chronos2_lora_training.ipynb` over the
12 official M6 rounds, and saves the raw quantile forecasts.

| | |
|---|---|
| Model | `amazon/chronos-2` plus the LoRA adapter trained on Dataset E |
| Output form | 21 quantile levels per asset and forecast day |
| Fixed | context 512 weekdays, horizon 20, `cross_learning=False`, official M6 asset order |
| Input | the same 12 shared round-context CSVs every other model used |
| Output | 12 quantile NPZ files, kept separate from all existing model outputs |

Everything about the inference path matches the zero-shot Chronos-2 run, so the
only difference between the two sets of forecasts is the adapter.

No scenario generation, no M6 probability conversion and no RPS: those are done
locally afterwards with the project's existing post-processing, once the 12
files have been copied back.

Run `chronos2_lora_training.ipynb` first. This notebook stops immediately if the
checkpoint is missing.

## 1. Install Chronos-2 and PEFT

In [ ]:
%pip install -q "chronos-forecasting==2.3.1" "transformers>=4.41,<5" "peft>=0.11"

In [ ]:
%pip uninstall -y torchao

## 2. Imports and settings

Identical to the zero-shot Chronos-2 run: same context length, horizon, seed,
asset order, round schedule and expected leading-NaN pattern.

In [ ]:
import json
import traceback
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from chronos import Chronos2Pipeline

BASE_MODEL_ID = "amazon/chronos-2"

CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 20
RANDOM_SEED = 42
ROUNDS = range(1, 13)

# Chronos-2 shares information across the variates of ONE task. cross_learning
# shares information across SEPARATE tasks, and this submits a single task, so
# it stays False exactly as in the zero-shot run.
CROSS_LEARNING = False

EXPECTED_QUANTILE_LEVELS = (
    [0.01, 0.05] + [round(0.05 * i, 2) for i in range(2, 19)] + [0.95, 0.99]
)

# Official M6 round schedule, unchanged.
# round -> context start, origin (= context end), forecast start, forecast end
ROUND_SCHEDULE = {
    1:  ("2020-03-19", "2022-03-04", "2022-03-07", "2022-04-01"),
    2:  ("2020-04-16", "2022-04-01", "2022-04-04", "2022-04-29"),
    3:  ("2020-05-14", "2022-04-29", "2022-05-02", "2022-05-27"),
    4:  ("2020-06-11", "2022-05-27", "2022-05-30", "2022-06-24"),
    5:  ("2020-07-09", "2022-06-24", "2022-06-27", "2022-07-22"),
    6:  ("2020-08-06", "2022-07-22", "2022-07-25", "2022-08-19"),
    7:  ("2020-09-03", "2022-08-19", "2022-08-22", "2022-09-16"),
    8:  ("2020-10-01", "2022-09-16", "2022-09-19", "2022-10-14"),
    9:  ("2020-10-29", "2022-10-14", "2022-10-17", "2022-11-11"),
    10: ("2020-11-26", "2022-11-11", "2022-11-14", "2022-12-09"),
    11: ("2020-12-24", "2022-12-09", "2022-12-12", "2023-01-06"),
    12: ("2021-01-21", "2023-01-06", "2023-01-09", "2023-02-03"),
}

# Genuine leading missing history, preserved as NaN for the model's own masking.
EXPECTED_LEADING_NAN = {r: {"OGN": 302 - 20 * (r - 1)} for r in ROUNDS}
EXPECTED_LEADING_NAN[1] = {"CARR": 1, "OGN": 302}

# Official M6 asset order - used to verify the contexts, not to reorder them.
OFFICIAL_ASSET_ORDER = [
    "ABBV", "ACN", "AEP", "AIZ", "ALLE", "AMAT", "AMP", "AMZN", "AVB", "AVY",
    "AXP", "BDX", "BF-B", "BMY", "BR", "CARR", "CDW", "CE", "CHTR", "CNC",
    "CNP", "COP", "CTAS", "CZR", "DG", "DPZ", "DRE", "DXC", "EWA", "EWC",
    "EWG", "EWH", "EWJ", "EWL", "EWQ", "EWT", "EWU", "EWY", "EWZ", "FTV",
    "GOOG", "GPC", "GSG", "HIG", "HIGH.L", "HST", "HYG", "IAU", "ICLN",
    "IEAA.L", "IEF", "IEFM.L", "IEMG", "IEUS", "IEVL.L", "IGF", "INDA",
    "IUMO.L", "IUVL.L", "IVV", "IWM", "IXN", "JPEA.L", "JPM", "KR", "LQD",
    "MCHI", "META", "MVEU.L", "OGN", "PG", "PPL", "PRU", "PYPL", "RE",
    "REET", "ROL", "ROST", "SEGA.L", "SHY", "SLV", "SPMV.L", "TLT", "UNH",
    "URI", "V", "VRSK", "VXX", "WRK", "XLB", "XLC", "XLE", "XLF", "XLI",
    "XLK", "XLP", "XLU", "XLV", "XLY", "XOM",
]
N_ASSETS = len(OFFICIAL_ASSET_ORDER)

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Versions")
print(f"  chronos-forecasting : {version('chronos-forecasting')}")
print(f"  transformers        : {version('transformers')}")
print(f"  torch               : {torch.__version__}")
print(f"\nLoRA fine-tuned {BASE_MODEL_ID}, M6 rounds {min(ROUNDS)}-{max(ROUNDS)}")
print(f"Model input per round : (1, {N_ASSETS}, {CONTEXT_LENGTH})")
print(f"Saved array per round : ({N_ASSETS}, {PREDICTION_LENGTH}, "
      f"{len(EXPECTED_QUANTILE_LEVELS)})")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Paths

Contexts come from the same shared folder every model has used. Outputs go to a
folder of their own, well away from the zero-shot Chronos-2 quantiles.

In [ ]:
# Existing shared context folder - the same 12 files as every previous run.
DRIVE_CONTEXT_DIR = Path("/content/drive/MyDrive/HonoursResearch/Round_1_Context")

# The checkpoint written by chronos2_lora_training.ipynb
LORA_CHECKPOINT_DIR = Path("/content/drive/MyDrive/HonoursResearch/checkpoints/chronos2_lora")

# Fine-tuned outputs, separate from every existing model's results
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/HonoursResearch/outputs/chronos2_lora/m6")

OUTPUT_PREFIX = "chronos2_lora"


def context_path(round_number: int) -> Path:
    return DRIVE_CONTEXT_DIR / f"round_{round_number:02d}_context.csv"


def quantiles_path(round_number: int) -> Path:
    return DRIVE_OUTPUT_DIR / f"{OUTPUT_PREFIX}_round{round_number:02d}_quantiles.npz"


if not DRIVE_CONTEXT_DIR.is_dir():
    raise FileNotFoundError(
        f"Context folder not found:\n  {DRIVE_CONTEXT_DIR}\n"
        "It must be the SAME folder used by the earlier model notebooks, "
        "containing round_01_context.csv .. round_12_context.csv."
    )

if not LORA_CHECKPOINT_DIR.is_dir() or not any(LORA_CHECKPOINT_DIR.iterdir()):
    raise FileNotFoundError(
        f"No fine-tuned Chronos-2 checkpoint at:\n  {LORA_CHECKPOINT_DIR}\n\n"
        "Run chronos2_lora_training.ipynb to completion first. That notebook "
        "fine-tunes the LoRA adapter on Dataset E and writes the checkpoint to "
        "this exact path. This notebook only runs inference and cannot create it."
    )

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Contexts (existing, shared) : {DRIVE_CONTEXT_DIR}")
print(f"LoRA checkpoint             : {LORA_CHECKPOINT_DIR}")
print(f"Output folder (new)         : {DRIVE_OUTPUT_DIR}")
print(f"\nExample output filename : {quantiles_path(1).name}")

metadata_file = LORA_CHECKPOINT_DIR / "training_metadata.json"
TRAINING_METADATA = (json.loads(metadata_file.read_text(encoding="utf-8"))
                     if metadata_file.is_file() else {})
if TRAINING_METADATA:
    print("\nCheckpoint was trained with:")
    for key in ("lora_rank", "lora_alpha", "learning_rate", "max_steps",
                "random_seed", "training_range", "validation_target"):
        if key in TRAINING_METADATA:
            print(f"  {key:<20} {TRAINING_METADATA[key]}")

## 5. Check the 12 context files are present

In [ ]:
missing = [context_path(r).name for r in ROUNDS if not context_path(r).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing context files in {DRIVE_CONTEXT_DIR}:\n  " + "\n  ".join(missing)
        + "\nThey must be the same round_NN_context.csv files used by every "
          "other model, copied from Data/contexts/."
    )
print(f"All {len(list(ROUNDS))} context files present in {DRIVE_CONTEXT_DIR}")

## 6. GPU check

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Runtime > Change runtime type > Hardware "
        "accelerator: GPU, then re-run from section 3."
    )

GPU_NAME = torch.cuda.get_device_name(0)
print(f"GPU: {GPU_NAME}")

## 7. Load the fine-tuned model, once

Loaded from the checkpoint and reused for all 12 rounds, never reloaded inside
the loop. The quantile levels are read from the loaded model and checked against
the documented native set, exactly as the zero-shot notebook does.

In [ ]:
pipeline = Chronos2Pipeline.from_pretrained(
    str(LORA_CHECKPOINT_DIR),
    device_map="cuda",
    import_allowlist=["chronos.chronos2.model"],
)

LOAD_ROUTE = "LoRA checkpoint loaded directly"

QUANTILE_LEVELS = np.asarray(pipeline.quantiles, dtype=np.float64)
N_QUANTILES = int(QUANTILE_LEVELS.size)

assert np.all(np.diff(QUANTILE_LEVELS) > 0), "Quantile levels are not increasing"
assert N_QUANTILES == len(EXPECTED_QUANTILE_LEVELS), (
    f"Expected {len(EXPECTED_QUANTILE_LEVELS)} native quantile levels, "
    f"the loaded model reports {N_QUANTILES}"
)

n_params = sum(p.numel() for p in pipeline.model.parameters())
print(f"Loaded: {LOAD_ROUTE}")
print(f"  from            : {LORA_CHECKPOINT_DIR}")
print(f"  parameters      : {n_params:,} ({n_params / 1e6:.1f}M)")
print(f"  quantile levels : {N_QUANTILES}")
print("  inference only - no further training happens here.")

## 8. Helpers

`load_and_validate_context` checks one round's shape, dates, asset order and
missing-value pattern. `forecast_round` submits the round as a single
multivariate task and inspects the returned layout rather than assuming it.
`save_and_verify_round` writes the NPZ and reads it back. `already_done` lets an
interrupted run resume safely.

In [ ]:
def load_and_validate_context(round_number: int):
    """Load one M6 context, validate it, and build the model input."""
    ctx_start, origin, fc_start, fc_end = ROUND_SCHEDULE[round_number]
    df = pd.read_csv(context_path(round_number), parse_dates=["date"])

    assert df.shape[0] == CONTEXT_LENGTH, (
        f"Round {round_number}: expected {CONTEXT_LENGTH} rows, found {df.shape[0]}"
    )
    assert list(df.columns) == ["date"] + OFFICIAL_ASSET_ORDER, (
        f"Round {round_number}: columns are not 'date' + the official M6 asset order"
    )
    dates = df["date"]
    assert dates.is_monotonic_increasing, f"Round {round_number}: dates not ascending"
    assert not dates.duplicated().any(), f"Round {round_number}: duplicate dates"
    assert dates.iloc[0] == pd.Timestamp(ctx_start), f"Round {round_number}: context start"
    assert dates.iloc[-1] == pd.Timestamp(origin), f"Round {round_number}: context end/origin"
    assert (dates <= pd.Timestamp(origin)).all(), (
        f"Round {round_number}: context contains a date after the origin"
    )

    matrix = df[OFFICIAL_ASSET_ORDER].to_numpy(dtype=np.float64).T
    assert matrix.shape == (N_ASSETS, CONTEXT_LENGTH)

    # Leading gaps are genuine later inceptions and stay NaN.
    leading_missing, interior_missing = {}, {}
    for i, symbol in enumerate(OFFICIAL_ASSET_ORDER):
        row = matrix[i]
        n_lead = int(np.argmax(~np.isnan(row))) if np.isnan(row[0]) else 0
        if n_lead:
            leading_missing[symbol] = n_lead
        if np.isnan(row[n_lead:]).any():
            interior_missing[symbol] = int(np.isnan(row[n_lead:]).sum())
    assert not interior_missing, (
        f"Round {round_number}: unexpected interior NaNs {interior_missing}"
    )
    assert leading_missing == EXPECTED_LEADING_NAN[round_number], (
        f"Round {round_number}: leading NaN pattern changed - "
        f"expected {EXPECTED_LEADING_NAN[round_number]}, found {leading_missing}"
    )

    forecast_dates = pd.bdate_range(fc_start, fc_end)
    assert len(forecast_dates) == PREDICTION_LENGTH, f"Round {round_number}: forecast window"
    return matrix, forecast_dates


def forecast_round(context_matrix, round_number: int):
    """Forecast one round as a single multivariate task.

    Input  : (1, 100, 512) - one task, 100 target variates, 512 observations.
    Output : (100, 20, 21) - assets x forecast days x native quantiles.

    The only change made to the context values is the float32 cast the tensor
    needs; NaNs are passed through for Chronos-2's own missing-value handling.
    """
    model_input = context_matrix[np.newaxis, :, :].astype(np.float32)
    assert model_input.shape == (1, N_ASSETS, CONTEXT_LENGTH)

    outputs = pipeline.predict(
        inputs=model_input,
        prediction_length=PREDICTION_LENGTH,
        cross_learning=CROSS_LEARNING,
    )
    assert len(outputs) == 1, (
        f"Round {round_number}: expected 1 task in the output, got {len(outputs)}"
    )
    raw = outputs[0]
    raw = raw.to(torch.float32).cpu().numpy() if torch.is_tensor(raw) else np.asarray(raw)

    # Inspect the returned layout instead of assuming it. 20 days and 21
    # quantiles differ in size, so the axes can be identified unambiguously.
    if raw.shape == (N_ASSETS, N_QUANTILES, PREDICTION_LENGTH):
        quantile_forecasts = np.transpose(raw, (0, 2, 1))
    elif raw.shape == (N_ASSETS, PREDICTION_LENGTH, N_QUANTILES):
        quantile_forecasts = raw
    else:
        raise ValueError(
            f"Round {round_number}: unrecognised output shape {raw.shape}; expected a "
            f"permutation of ({N_ASSETS}, {PREDICTION_LENGTH}, {N_QUANTILES})"
        )

    quantile_forecasts = np.ascontiguousarray(quantile_forecasts)
    assert quantile_forecasts.shape == (N_ASSETS, PREDICTION_LENGTH, N_QUANTILES)
    assert np.isfinite(quantile_forecasts).all(), (
        f"Round {round_number}: forecasts contain NaN or infinite values"
    )

    # Quantiles must not cross, for every asset and every forecast day.
    max_violation = float(-np.min(np.diff(quantile_forecasts, axis=2)))
    assert max_violation <= 1e-6, (
        f"Round {round_number}: quantiles are not monotonically ordered "
        f"(largest crossing {max_violation:.3e})"
    )
    return quantile_forecasts


def save_and_verify_round(round_number: int, quantile_forecasts, forecast_dates):
    """Save the raw quantile forecasts, then reload and verify the file."""
    path = quantiles_path(round_number)
    ctx_start, origin, fc_start, fc_end = ROUND_SCHEDULE[round_number]
    date_strings = np.array([d.strftime("%Y-%m-%d") for d in forecast_dates])

    # Same four arrays as the zero-shot Chronos-2 files, plus run metadata so a
    # fine-tuned file can never be mistaken for a zero-shot one.
    run_metadata = {
        "base_model_id": BASE_MODEL_ID,
        "adapter": "lora",
        "checkpoint": str(LORA_CHECKPOINT_DIR),
        "round": round_number,
        "origin": origin,
        "forecast_start": fc_start,
        "forecast_end": fc_end,
        "context_length": CONTEXT_LENGTH,
        "prediction_length": PREDICTION_LENGTH,
        "cross_learning": CROSS_LEARNING,
        "random_seed": RANDOM_SEED,
        "lora_rank": TRAINING_METADATA.get("lora_rank"),
        "lora_alpha": TRAINING_METADATA.get("lora_alpha"),
        "max_steps": TRAINING_METADATA.get("max_steps"),
        "generated_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
    }

    np.savez_compressed(
        path,
        quantile_forecasts=quantile_forecasts,
        quantile_levels=QUANTILE_LEVELS,
        asset_symbols=np.array(OFFICIAL_ASSET_ORDER),
        forecast_dates=date_strings,
        run_metadata=np.array(json.dumps(run_metadata)),
    )

    with np.load(path, allow_pickle=False) as reloaded:
        r_forecasts = reloaded["quantile_forecasts"]
        r_levels = reloaded["quantile_levels"]
        r_symbols = list(reloaded["asset_symbols"])
        r_dates = list(reloaded["forecast_dates"])

    assert r_forecasts.shape == (N_ASSETS, PREDICTION_LENGTH, N_QUANTILES), "reloaded shape"
    assert np.array_equal(r_forecasts, quantile_forecasts), "saved values differ"
    assert np.isfinite(r_forecasts).all(), "reloaded values not finite"
    assert np.allclose(r_levels, QUANTILE_LEVELS), "quantile levels changed on save/reload"
    assert r_symbols == OFFICIAL_ASSET_ORDER, "asset ordering changed on save/reload"
    assert r_dates == list(date_strings), "forecast dates changed on save/reload"
    return path


def already_done(round_number: int) -> bool:
    """True if this round is saved and passes the essential checks.

    Used for resume after a disconnect. A file that fails any check, or is
    unreadable, counts as not done and is regenerated.
    """
    path = quantiles_path(round_number)
    if not path.is_file():
        return False
    _, _, fc_start, fc_end = ROUND_SCHEDULE[round_number]
    expected_dates = [d.strftime("%Y-%m-%d") for d in pd.bdate_range(fc_start, fc_end)]
    try:
        with np.load(path, allow_pickle=False) as f:
            if not {"quantile_forecasts", "quantile_levels",
                    "asset_symbols", "forecast_dates"} <= set(f.files):
                return False
            if f["quantile_forecasts"].shape != (N_ASSETS, PREDICTION_LENGTH, N_QUANTILES):
                return False
            if list(f["asset_symbols"]) != OFFICIAL_ASSET_ORDER:
                return False
            if list(f["forecast_dates"]) != expected_dates:
                return False
            return bool(np.isfinite(f["quantile_forecasts"]).all())
    except Exception:
        return False


print("Helpers defined.")

### Run all 12 rounds

Each round is written and verified before the next begins. Re-running this cell
resumes: rounds already saved and passing their checks are skipped.

In [ ]:
records, failures, skipped = [], [], []
experiment_started = datetime.now(timezone.utc)

for r in ROUNDS:
    _, origin, fc_start, fc_end = ROUND_SCHEDULE[r]
    print(f"\n{'=' * 70}\nRound {r:02d} | origin {origin} | "
          f"forecast {fc_start} .. {fc_end}\n{'=' * 70}")

    if already_done(r):
        skipped.append(r)
        print("  already saved and verified, skipped")
        continue

    try:
        context_matrix, forecast_dates = load_and_validate_context(r)

        started = datetime.now(timezone.utc)
        quantile_forecasts = forecast_round(context_matrix, r)
        elapsed = (datetime.now(timezone.utc) - started).total_seconds()

        saved_path = save_and_verify_round(r, quantile_forecasts, forecast_dates)
        records.append({"round": r, "origin": origin, "seconds": round(elapsed, 1),
                        "file": saved_path.name})
        print(f"  forecast {tuple(quantile_forecasts.shape)} in {elapsed:.1f}s")
        print(f"  saved+verified: {saved_path.name}")

        del quantile_forecasts, context_matrix
        torch.cuda.empty_cache()

    except Exception as exc:
        failures.append((r, f"{type(exc).__name__}: {exc}"))
        print(f"  !! ROUND {r:02d} FAILED - completed rounds remain saved")
        traceback.print_exc()
        torch.cuda.empty_cache()

total_minutes = (datetime.now(timezone.utc) - experiment_started).total_seconds() / 60
print(f"\nFinished in {total_minutes:.1f} min. "
      f"Generated {len(records)}, skipped {len(skipped)}, failed {len(failures)} "
      f"of {len(list(ROUNDS))} rounds.")
if failures:
    print(f"FAILED rounds: {[r for r, _ in failures]}")

## 9. Verify all 12 saved files

Re-reads every file from Drive, independently of the loop above.

In [ ]:
ALL_VERIFIED = True
print(f"{'rnd':>4}  {'file':<50} {'status':<8} shape")

for r in ROUNDS:
    path = quantiles_path(r)
    if not path.is_file():
        ALL_VERIFIED = False
        print(f"{r:>4}  {path.name:<50} MISSING")
        continue

    _, _, fc_start, fc_end = ROUND_SCHEDULE[r]
    expected_dates = [d.strftime("%Y-%m-%d") for d in pd.bdate_range(fc_start, fc_end)]
    with np.load(path, allow_pickle=False) as f:
        forecasts = f["quantile_forecasts"]
        levels = f["quantile_levels"]
        symbols = list(f["asset_symbols"])
        fdates = list(f["forecast_dates"])

    ok = (forecasts.shape == (N_ASSETS, PREDICTION_LENGTH, N_QUANTILES)
          and symbols == OFFICIAL_ASSET_ORDER
          and fdates == expected_dates
          and np.allclose(levels, QUANTILE_LEVELS)
          and bool(np.isfinite(forecasts).all()))
    ALL_VERIFIED = ALL_VERIFIED and ok
    print(f"{r:>4}  {path.name:<50} {'OK' if ok else 'FAILED':<8} {forecasts.shape}")

print(f"\nAll {len(list(ROUNDS))} files present and verified: {ALL_VERIFIED}")

## 10. Completion summary

In [ ]:
print("Chronos-2 LoRA - M6 rounds 1-12")
print(f"  base model : {BASE_MODEL_ID} plus LoRA adapter")
print(f"  checkpoint : {LORA_CHECKPOINT_DIR}")
print(f"  loaded via : {LOAD_ROUTE}")
print(f"  fixed      : context {CONTEXT_LENGTH}, horizon {PREDICTION_LENGTH}, "
      f"cross_learning {CROSS_LEARNING}, {N_ASSETS} assets")
print(f"  rounds     : {len(records)} generated, {len(skipped)} skipped, "
      f"{len(failures)} failed")
print(f"  verified   : {ALL_VERIFIED}")
print(f"  output     : {DRIVE_OUTPUT_DIR}")
print("  post-processing: none here (no scenarios, no quintiles, no RPS)")

saved = sum(1 for r in ROUNDS if quantiles_path(r).is_file())
if saved == len(list(ROUNDS)):
    print("\n  All 12 rounds COMPLETE. Copy the folder from Drive into")
    print("  TSFM Uncertainty Experiment/model_outputs/chronos2_lora/m6/")
    print("  then score locally with the project's existing Chronos-2 "
          "quantile-to-scenario and RPS code.")
else:
    print(f"\n  INCOMPLETE: {saved}/12 rounds saved. Re-run the loop in section 8 "
          "to resume; finished rounds are skipped.")